In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tkinter as tk

from tkinter import messagebox

In [3]:
df = pd.read_csv('pricerunner_aggregate.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Product ID       35311 non-null  int64
 1   Product Title    35311 non-null  str  
 2    Merchant ID     35311 non-null  int64
 3    Cluster ID      35311 non-null  int64
 4    Cluster Label   35311 non-null  str  
 5    Category ID     35311 non-null  int64
 6    Category Label  35311 non-null  str  
dtypes: int64(4), str(3)
memory usage: 1.9 MB


In [5]:
# df = df.drop('Product ID', axis=1)

In [47]:
df.sample()

,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
8692,14820,intel xeon 6148 processor 2.4 ghz box 27.5 mb l3,64,6077,Intel Xeon Gold 6148 2.4GHz Box,2615,3


In [7]:
df[' Cluster Label'].value_counts()

 Cluster Label
Canon IXUS 185              27
Samsung UE49NU7100          24
Canon PowerShot SX730 HS    24
Apple iPhone 8 Plus 64GB    23
Samsung UE65NU7100          23
                            ..
Smeg FAB28 Cream             1
Smeg FAB28 Red               1
Smeg FAB28 Pink              1
Candy CRU16.0                1
Neff K4316                   1
Name: count, Length: 12849, dtype: int64

In [13]:
df[' Category Label'].value_counts()

 Category Label
Fridge Freezers     5501
Mobile Phones       4081
Washing Machines    4044
CPUs                3862
Fridges             3584
TVs                 3564
Dishwashers         3424
Digital Cameras     2697
Microwaves          2342
Freezers            2212
Name: count, dtype: int64

In [15]:
df[' Cluster ID'].value_counts()

 Cluster ID
38848    27
4419     24
38849    24
1        23
4420     23
         ..
47517     1
47518     1
47519     1
47524     1
47525     1
Name: count, Length: 13233, dtype: int64

### Model

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 35311 entries, 0 to 35310
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Product ID       35311 non-null  int64
 1   Product Title    35311 non-null  str  
 2    Merchant ID     35311 non-null  int64
 3    Cluster ID      35311 non-null  int64
 4    Cluster Label   35311 non-null  str  
 5    Category ID     35311 non-null  int64
 6    Category Label  35311 non-null  str  
dtypes: int64(4), str(3)
memory usage: 1.9 MB


In [38]:
df[' Category Label'].value_counts()

 Category Label
0    5501
1    4081
2    4044
3    3862
4    3584
5    3564
6    3424
7    2697
8    2342
9    2212
Name: count, dtype: int64

In [26]:
df[' Category Label'] = df[' Category Label'].map(
    {'Fridge Freezers':0,
    'Mobile Phones':1,
    'Washing Machines':2,
    'CPUs':3,
    'Fridges':4,
    'TVs':5,
    'Dishwashers':6,
    'Digital Cameras':7,
    'Microwaves':8,
    'Freezers':9,}
)

In [37]:
aa = df[[' Cluster Label', 'Product Title']]
pd.factorize(aa)

TypeError: factorize requires a Series, Index, ExtensionArray, np.ndarray or NumpyExtensionArray got DataFrame.

In [28]:
X = df.drop([' Category Label', ' Category ID', 'Product ID'], axis=1)
y = df[' Category Label']

In [11]:
def split_data(X, y, size=0.8):
    np.random.seed(42)

    train_idx, test_idx = [], []

    for i in np.unique(y):
        idx = np.where(y == i)
        np.random.permutation(idx)
        split = int(len(idx) * size)
        
        train_idx.extend(idx[:split])
        test_idx.extend(idx[split:])

    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (10,) + inhomogeneous part.

In [ ]:
class Node:
    def __init__(self):
        self.pred_class = None
        self.feature = None
        self. threshold = None
        self.left = None
        self.right = None

class DTCART:
    def __init__(self, max_depth=3, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_split = min_samples_split
        self.min_leaf = min_samples_leaf
        self.root = None
        self.classes = None

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes = np.unique(y)
        self.root = self._grow(X, y, 0)

        return self
    
    def _gini(self, y):
        if len(y == 0):
            return 0
        probs = np.bincount(y, minlength=len(self.classes)) / len(y)
        return 1 - np.sum(probs ** 2)
    
    def _split(self, X, y):
        m, n = X.shape

        if m < self.min_split:
            return None, None

        best_gini = self._gini(y)
        best_feature, best_threshold = None, None

        for feature in range(n):
            idx = np.argsort(X[:, feature])
            x_sorted, y_sorted = X[idx, feature], y[idx]

            left_counts = np.zeros(len(self.classes), dtype=int)
            right_counts = np.bincount(y_sorted, minlength=len(self.classes))

            for i in range(1, m):
                c = y_sorted[i-1]
                left_counts[c] += 1
                right_counts[c] += 1

                if x_sorted[i] == x_sorted[i-1]:
                    continue

                if i < self.min_leaf or (m - i) < self.min_leaf:
                    continue

                left_gini = 1 - np.sum((left_counts / i)**2)
                right_gini = 1 - np.sum((right_counts / (m-i))**2)
                gini = (i * left_gini + (m-1) * right_gini) / m

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = (x_sorted[i] + x_sorted[i-1]) / 2

        return best_feature, best_threshold

    def _grow(self, X, y, depth):
        node = Node()
        node.pred_class = np.bincount(y).argmax()

        if (self.max_depth and depth >= self.max_depth) or len(y) < self.min_split or len(np.unique(y)) == 1:
            return node

        feature, threshold = self._split(X, y)
        if feature is None:
            return node

        left_mask = X[:, feature] <= threshold
        if np.sum(left_mask) < self.min_leaf or np.sum(~left_mask) < self.min_leaf:
            return node

    
        node.feature = feature
        node.threshold = threshold
        node.left = self._grow(X[left_mask], y[left_mask], depth=1)
        node.right = self._grow(X[~left_mask], y[~left_mask], depth=1)

        return node
    
    def predict(self, X):
        X = np.asarray(X)
        return np.array([self._predict_one(x) for x in X])

    def _predict_one(self, x):
        node = self.root

        while node.left is not None:
            node = node.left if x[node.feature] <= node.threshold else node.right
        
        return self.classes[node.pred_class]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

In [ ]:
best_acc = 0
best_depth = None
accuracy = []

for depth in [2,3,4,5,6,7,8,9,10,11]:
    model = DTCART(max_depth=depth)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = model.score(pred == y_test)

    print(f'Depth: {depth} - Accuracy: {acc:.4f}')

    if acc > best_acc:
        best_acc = acc
        best_depth = depth
print(f'Best Depth: {best_depth} - Accuracy: {best_acc:.4f}')

NameError: name 'X_train' is not defined

In [ ]:
model = DTCART(max_depth=best_depth)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
model.score(y_pred_train, y_train)
model.score(y_pred_test, y_test)

### Evaluasi

In [ ]:
def evaluate(y_true, y_pred, title='Train'):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    for i in np.unique(y):
        tp = np.sum((y_true == i) & (y_pred == i))
        fp = np.sum((y_true != i) & (y_pred == i))
        fn = np.sum((y_true == i) & (y_pred != i))

        accuracy = np.mean(y_true == y_pred)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f'Evaluasi {title}')
    print(f'Accuracy    : {accuracy}')
    print(f'Precision   : {precision}')
    print(f'Recall      : {recall}')
    print(f'F1          : {f1}')

In [ ]:
evaluate(y_train, y_pred_train)
evaluate(y_test, y_pred_test, title='Test')

In [ ]:
def cmtrx(y_true, y_pred, title='Train'):
    classes = np.unique(y_true)
    n_classes = len(classes)

    cm = np.zeros((n_classes, n_classes))

    for i, c_true in enumerate(classes):
        for j, c_pred in enumerate(classes):
            cm[i,j] = np.sum((y_true == c_true) * (y_pred == c_pred))

    plt.figure(figsize=(10,7))
    plt.title(f'Confusion Matrix {title}', fontsize=15)
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', cbar=False, xticklabels=classes, yticklabels=classes)
    plt.xlabel('Prediksi')
    plt.ylabel('Aktual')
    plt.show()

In [ ]:
cmtrx(y_train, y_pred_train)
cmtrx(y_test, y_pred_test, title='Test')

In [ ]:
def visualize(tree, feature_name, depth=0, x=0.5, y=1.0, dx=0.3, parent_x=None):
    box_props = dict(
        boxstyle = 'round, pad=0.5',
        facecolor = 'lightgreen' if (isinstance(tree, str) or (not isinstance(tree, dict) and tree.left is None and tree.right is None)) else 'lightblue'
    )

    if isinstance(tree, str):
        plt.text(x, y, tree, ha='center', va='center', fontsize=6, bbox=box_propsm, zorder=3)
        return

    if isinstance(tree, dict):
        feature_idx = tree['feature_idx']
        val = tree['val']
        left = tree['left']
        right = tree['right']
        gini = tree.gini('gini', '')
        samples = tree.gini('sample', '')

    else:
        if tree.left is None and tree.right is None:
            label = f'Class {tree.predicted_class}\nGini: {tree.gini:.3f}\nSample: {tree.num_samples}'
            plt.text(x, y, label, ha='center', va='center', fontsize=7, bbox=box_props, zorder=3)

        feature_idx = tree.feature_idx
        val = tree.threshold
        left = tree.left
        right = tree.right
        gini = tree.gini
        samples = tree.num_samples

    feature = feature_name[feature_idx] if feature_idx < len(feature_name) else f'Feature[{feature_idx}]'
    label = f'{feature} < {val:.2f}\nGini: {gini:.3f}\nSamples: {samples}'

    plt.text(x, y, label, ha='center', va='center', fontsize=7, bbox=box_props, zorder=3)

    next_y = y - 0.15
    next_dx = dx / (1.2 ** depth)

    left_x = x - next_dx
    right_x = x + next_dx

    margin = 0.03
    plt.plot([x, left_x], [y - margin, next_y + margin], 'k-', linewidth=1.2, zorder=1)
    plt.plot([x, right_x], [y - margin, next_y + margin], 'k-', linewidth=1.2, zorder=1)

    margin = 0.03
    visualize(left, feature_name, depth + 1, left_x, next_y, dx)
    visualize(right, feature_name, depth + 1, right_x, next_y, dx)

plt.figure(figsize=(14,11))
visualize(model.tree_, X.columns.to_list(), dx=0.4)
plt.show()